# 2-2 Lite. Great Expectations 결과 읽기 실습

이 notebook은 JupyterLite에서 Great Expectations(GE) 결과를 읽고 해석하는 실습입니다.

브라우저 환경에서는 GE 패키지를 직접 실행하지 않습니다. 대신 로컬에서 미리 만든 prepared artifact를 읽습니다.

| 파일 | 쉬운 뜻 | 이 notebook에서 하는 일 |
| --- | --- | --- |
| `chapter_02_expectations.json` | 검사 기준 목록 | 어떤 검사를 하기로 했는지 봅니다. |
| `chapter_02_validation_result.json` | 검사 실행 결과 | 어떤 검사가 통과/실패했는지 봅니다. |
| `chapter_02_validation_summary.md` | 사람이 읽는 요약 | 보고서에 인용할 문장을 확인합니다. |
| `chapter_02_data_docs.html` | GE Data Docs 예시 | GE가 HTML 문서로도 결과를 보여줄 수 있음을 확인합니다. |

목표는 GE API를 실행하는 것이 아니라, **GE 결과가 모델 평가 전에 어떤 데이터 품질 신호를 주는지 읽는 것**입니다.


### 따라하기 안내: GE artifact 읽기 흐름

1. prepared artifact 경로를 확인합니다.
2. expectation JSON에서 검사 기준을 봅니다.
3. validation result JSON에서 전체 성공 여부를 봅니다.
4. 실패한 검사만 따로 확인합니다.
5. summary markdown 일부를 읽고 보고서 표현으로 연결합니다.
6. 다음 모델 평가/데이터-지표 연결 notebook으로 넘어갑니다.

중요한 점은 “GE를 실행했다”가 아니라 “준비된 GE 결과를 확인했다”라고 구분하는 것입니다.


## 1. 실행 준비

### 1-1. 필요한 도구와 artifact 경로 확인

이 셀에서는 JSON과 Markdown 파일을 읽기 위한 기본 도구를 준비합니다. 아직 파일 내용은 읽지 않습니다.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

def resolve_artifact_path(relative_path: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


EXPECTATIONS_PATH = resolve_artifact_path("artifacts/great_expectations/chapter_02_expectations.json")
VALIDATION_RESULT_PATH = resolve_artifact_path("artifacts/great_expectations/chapter_02_validation_result.json")
VALIDATION_SUMMARY_PATH = resolve_artifact_path("artifacts/great_expectations/chapter_02_validation_summary.md")
DATA_DOCS_PATH = resolve_artifact_path("artifacts/great_expectations/chapter_02_data_docs.html")

ge_artifact_paths = pd.DataFrame(
    [
        {"artifact": "expectations", "path": str(EXPECTATIONS_PATH), "exists": EXPECTATIONS_PATH.exists(), "easy_meaning": "검사 기준 목록"},
        {"artifact": "validation_result", "path": str(VALIDATION_RESULT_PATH), "exists": VALIDATION_RESULT_PATH.exists(), "easy_meaning": "검사 실행 결과"},
        {"artifact": "validation_summary", "path": str(VALIDATION_SUMMARY_PATH), "exists": VALIDATION_SUMMARY_PATH.exists(), "easy_meaning": "사람이 읽는 요약"},
        {"artifact": "data_docs", "path": str(DATA_DOCS_PATH), "exists": DATA_DOCS_PATH.exists(), "easy_meaning": "HTML 결과 문서"},
    ]
)


**출력 확인: `ge_artifact_paths`**

네 파일이 모두 `exists=True`인지 확인합니다. 하나라도 없으면 JupyterLite 배포 자료에 GE 결과 파일이 빠진 것입니다.


In [ ]:
display(ge_artifact_paths)

## 2. 검사 기준 읽기

### 2-1. expectation 목록 확인

Expectation은 검사 항목입니다. 예를 들어 “`label` 컬럼이 있어야 한다”, “`heart_rate`는 비어 있으면 안 된다” 같은 규칙입니다.


In [ ]:
expectations_payload = json.loads(EXPECTATIONS_PATH.read_text(encoding="utf-8"))
expectation_table = pd.DataFrame(expectations_payload["expectations"])
expectation_table = expectation_table.assign(
    easy_type=lambda table: table["expectation_type"].map(
        {
            "expect_column_to_exist": "컬럼 존재 확인",
            "expect_column_values_to_not_be_null": "결측 확인",
            "expect_column_values_to_be_in_set": "허용값 확인",
            "expect_column_values_to_be_between": "범위 확인",
        }
    )
)


**출력 확인: `expectation_table`**

이 표는 검사 결과가 아니라 검사 기준입니다. 어떤 컬럼을 어떤 규칙으로 볼 것인지 먼저 확인합니다.


In [ ]:
display(expectation_table.loc[:, ["expectation_type", "easy_type", "column", "qa_reason"]])

## 3. 검사 결과 읽기

### 3-1. 전체 성공 여부 확인

이제 validation result JSON을 읽습니다. 먼저 전체 행 수, 통과 검사 수, 실패 검사 수를 봅니다.


In [ ]:
validation_payload = json.loads(VALIDATION_RESULT_PATH.read_text(encoding="utf-8"))
validation_overview = pd.DataFrame(
    [
        {"item": "dataset_name", "value": validation_payload["dataset_name"], "easy_meaning": "검사한 데이터 파일"},
        {"item": "row_count", "value": validation_payload["row_count"], "easy_meaning": "검사 대상 행 수"},
        {"item": "success", "value": validation_payload["success"], "easy_meaning": "전체 검사 통과 여부"},
        {"item": "success_count", "value": validation_payload["success_count"], "easy_meaning": "통과한 검사 수"},
        {"item": "failure_count", "value": validation_payload["failure_count"], "easy_meaning": "실패한 검사 수"},
    ]
)
validation_result_table = pd.DataFrame(validation_payload["expectation_results"])
validation_result_table = validation_result_table.assign(
    status=lambda table: table["success"].map({True: "pass", False: "fail"}),
    unexpected_ratio_pct=lambda table: table["unexpected_ratio"].round(2),
)


**출력 확인: `validation_overview`**

`success=False`이면 하나 이상의 검사 항목이 실패했다는 뜻입니다. 이번 데이터에서는 실패가 있는 상태가 정상적인 실습 흐름입니다.


In [ ]:
display(validation_overview)

### 3-2. 검사별 통과/실패 확인

전체 요약 다음에는 각 검사 항목별 결과를 봅니다.


**출력 확인: `validation_result_table`**

`status`가 `fail`인 행이 핵심입니다. `unexpected_count`는 규칙을 어긴 행 수입니다.


In [ ]:
display(
    validation_result_table.loc[:, [
        "expectation_type",
        "column",
        "status",
        "unexpected_count",
        "unexpected_ratio_pct",
        "observed_value",
        "qa_reason",
    ]]
)

### 3-3. 실패한 검사만 따로 보기

보고서에 가장 먼저 인용할 부분은 실패한 검사입니다.


In [ ]:
ge_failure_table = validation_result_table.loc[
    ~validation_result_table["success"],
    ["expectation_type", "column", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason"],
].reset_index(drop=True)


**출력 확인: `ge_failure_table`**

이번 실습에서는 `heart_rate` 결측과 `oxygen_saturation` 범위 오류가 핵심 품질 신호입니다. 이 신호는 뒤의 모델 지표 해석에서 원인 후보로 연결됩니다.


In [ ]:
display(ge_failure_table)

## 4. Markdown 요약 확인

### 4-1. summary 파일 앞부분 읽기

JSON은 계산용이고, Markdown summary는 사람이 보고서에 옮기기 쉬운 형태입니다.


In [ ]:
validation_summary_text = VALIDATION_SUMMARY_PATH.read_text(encoding="utf-8")
validation_summary_preview = "\n".join(validation_summary_text.splitlines()[:14])


**출력 확인: `validation_summary_preview`**

Markdown 요약의 앞부분입니다. 데이터셋 이름, 행 수, 실패 조건 수를 빠르게 확인합니다.


In [ ]:
print(validation_summary_preview)

## 5. 다음 notebook으로 연결

### 5-1. GE 결과를 모델 평가 해석으로 넘기기

이 notebook에서 확인한 핵심은 다음과 같습니다.

| 확인한 것 | 해석 |
| --- | --- |
| `label` 컬럼 존재/결측/허용값 기준은 통과 | 지표 계산 형식은 유지됩니다. |
| `heart_rate` 결측 실패 | score 계산 입력 품질 이슈 후보입니다. |
| `oxygen_saturation` 범위 실패 | score 분포 왜곡 후보입니다. |
| 전체 GE 결과는 실패 | 모델 지표를 해석할 때 데이터 품질 제한을 함께 적어야 합니다. |

다음에는 `06_model_evaluation_lab.ipynb`에서 score, threshold, metric을 보고, `07_data_metric_connection_lab.ipynb`에서 이 GE 실패 신호와 metric 변화를 연결합니다.
